# Task 2: Automated Model Training Pipeline
**SalaryInsight: Predictive Modeling for Informed Salary Predictions**

Goal: combine preprocessing, hyperparameter tuning and performance evaluation into a single, reusable workflow.

In [1]:
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

RS = 42
LABELS = ["Low", "Medium", "High"]
import joblib
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

## 1. Pipeline building blocks

In [2]:
def load_data(path="employee_data.csv"):
    df = pd.read_csv(path)
    y = pd.qcut(df["Salary_LPA"], 3, labels=LABELS)        # salary bands
    return df.drop(columns=["Salary_LPA"]), y

def build_preprocessor(X):
    num = X.select_dtypes("number").columns.tolist()
    cat = [c for c in X.columns if c not in num]
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])

SEARCH_SPACES = {   # model + hyperparameter grid
    "LogisticRegression": (LogisticRegression(max_iter=3000),
                           {"clf__C": [0.01, 0.1, 1, 10, 100]}),
    "SVM": (SVC(random_state=RS),
            {"clf__kernel": ["linear", "rbf"], "clf__C": [0.1, 1, 10], "clf__gamma": ["scale", 0.01, 0.1]}),
    "RandomForest": (RandomForestClassifier(random_state=RS),
                     {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 10, 20],
                      "clf__min_samples_leaf": [1, 3]}),
}

## 2. Single workflow: preprocess → tune → evaluate → select → save

In [3]:
def run_pipeline():
    X, y = load_data()
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)
    pre = build_preprocessor(X)
    cv = StratifiedKFold(5, shuffle=True, random_state=RS)
    results, fitted = [], {}
    for name, (clf, grid) in SEARCH_SPACES.items():
        gs = GridSearchCV(Pipeline([("pre", pre), ("clf", clf)]), grid, cv=cv,
                          scoring="f1_macro", n_jobs=-1).fit(Xtr, ytr)
        p = gs.predict(Xte); fitted[name] = gs.best_estimator_
        results.append({"Model": name, "Best CV F1": round(gs.best_score_, 4),
                        "Test Acc": round(accuracy_score(yte, p), 4),
                        "Test F1": round(f1_score(yte, p, average="macro"), 4),
                        "Best Params": gs.best_params_})
    res = pd.DataFrame(results).sort_values("Best CV F1", ascending=False).reset_index(drop=True)
    best = res.loc[0, "Model"]
    return res, best, fitted[best], Xte, yte

In [4]:
results, best_name, best_model, Xte, yte = run_pipeline()
results

,Model,Best CV F1,Test Acc,Test F1,Best Params
0,SVM,0.8689,0.8725,0.8736,"{'clf__C': 10, 'clf__gamma': 0.01, 'clf__kerne..."
1,LogisticRegression,0.8662,0.8825,0.8827,{'clf__C': 100}
2,RandomForest,0.8134,0.8350,0.8347,"{'clf__max_depth': None, 'clf__min_samples_lea..."


## 3. Evaluate the selected model

In [5]:
p = best_model.predict(Xte)
print("Selected model:", best_name)
print(classification_report(yte, p, labels=LABELS))
pd.DataFrame(confusion_matrix(yte, p, labels=LABELS),
             index=[f"actual {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS])

Selected model: SVM
              precision    recall  f1-score   support

         Low       0.93      0.92      0.92       134
      Medium       0.78      0.86      0.82       133
        High       0.93      0.83      0.88       133

    accuracy                           0.87       400
   macro avg       0.88      0.87      0.87       400
weighted avg       0.88      0.87      0.87       400



,pred Low,pred Medium,pred High
actual Low,123,11,0
actual Medium,9,115,9
actual High,0,22,111


## 4. Save the trained pipeline
The saved object contains preprocessing + model, so it can predict directly from raw employee data.

In [6]:
joblib.dump(best_model, "best_salary_model.joblib")
print("Saved best_salary_model.joblib")

Saved best_salary_model.joblib


**Result:** tuned SVM (RBF, C=10, gamma=0.01) is selected on cross-validated macro-F1 (0.869) with ~87% test accuracy. Tuned Logistic Regression is statistically tied (0.866 CV, 0.883 test).